# FleetGuard AI — Simple ML Workflow
AI4I 2020 predictive maintenance classification.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
df = pd.read_csv('../data/ai4i2020.csv')
df.shape, df.isnull().sum().sum(), df.duplicated().sum()


## Features and target
Failure-mode columns are excluded to avoid target leakage.


In [ ]:
features = ['Type','Air temperature [K]','Process temperature [K]','Rotational speed [rpm]','Torque [Nm]','Tool wear [min]']
X = df[features]
y = df['Machine failure']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
num = features[1:]
cat = ['Type']
preprocessor = ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])


In [ ]:
models = {
 'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
 'Decision Tree': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
 'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42),
 'SVM': SVC(probability=True, class_weight='balanced', random_state=42)
}
for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    print('\n', name)
    print(classification_report(y_test, pipe.predict(X_test)))


## Deployment
The selected saved pipeline is loaded by the Streamlit app.
